# DS 227 · Week 6 — Data preprocessing II
## Integration, transformation and reduction

Every code block from the week 6 lecture, in order, so you can run what you watched
instead of retyping it. The first slides use small tables typed into the cells; the
**Case study** cell loads the flood control data from the course API, so there is no file
to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 8 — Code on these slides runs in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 10 — Stacking rows from tables with the same columns


Expected output:

```
jan:
    city  sales
0   Cebu     10
1  Davao     22

feb:
     city  sales
0    Cebu     14
1  Iloilo      9

both:
     city  sales
0    Cebu     10
1   Davao     22
2    Cebu     14
3  Iloilo      9
```


In [ ]:
jan = pd.DataFrame({"city": ["Cebu", "Davao"], "sales": [10, 22]})
feb = pd.DataFrame({"city": ["Cebu", "Iloilo"], "sales": [14, 9]})

print("jan:", jan, sep="\n")
print("\nfeb:", feb, sep="\n")

both = pd.concat([jan, feb], ignore_index=True)
print("\nboth:", both, sep="\n")

### Slide 11 — Joining tables with different facts on a key


**Exercise.**

1. Predict Davao’s region after the merge.
2. Run the cell.
3. Change how="left" to how="inner". Which row disappears?


In [ ]:
sales = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo"],
                      "sales": [24, 22, 9]})
meta = pd.DataFrame({"city": ["Cebu", "Iloilo", "Baguio"],
                     "region": ["Visayas", "Visayas", "Luzon"]})

print("sales:", sales, sep="\n")
print("\nmeta:", meta, sep="\n")

merged = pd.merge(sales, meta, on="city", how="left")
print("\nmerged:", merged, sep="\n")

### Slide 13 — Join types and the question each answers


Expected output:

```
3
2
city  sales region     _merge
1  Davao     22    NaN  left_only
```


In [ ]:
# sales, meta: the two tables from the join exercise
print(len(pd.merge(sales, meta, on="city", how="left")))
print(len(pd.merge(sales, meta, on="city", how="inner")))

m = pd.merge(sales, meta, on="city", how="left", indicator=True)
print(m[m._merge == "left_only"])

### Slide 14 — A repeated key multiplies rows


Expected output:

```
False
3 rows -> 4 rows
79 instead of 55
```


In [ ]:
# Cebu listed twice in the lookup table
dup = pd.DataFrame({"city": ["Cebu", "Cebu", "Iloilo"],
                    "region": ["Visayas", "Visayas", "Visayas"]})
print(dup.city.is_unique)

merged = pd.merge(sales, dup, on="city", how="left")
print(len(sales), "rows ->", len(merged), "rows")
print(merged.sales.sum(), "instead of", sales.sales.sum())

### Slide 15 — Combining sources creates duplicates


Expected output:

```
city  year  sales
0   Cebu  2024     24
2  Davao  2024     22
3   Cebu  2025     30
```


In [ ]:
records = pd.DataFrame({"city": ["Cebu", "Cebu", "Davao", "Cebu"],
                        "year": [2024, 2024, 2024, 2025],
                        "sales": [24, 24, 22, 30]})

clean = records.drop_duplicates()
print(clean)

### Slide 16 — The flood data, loaded from the course API


Expected output:

```
(34079, 15)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

print(df.shape)

### Slide 17 — Verifying that the key is unique


Expected output:

```
34079 34079
(0, 15)
```


In [ ]:
print(df.contract_id.nunique(), len(df))

# if they were not equal, list the repeats:
repeats = df[df.contract_id.duplicated(keep=False)]
print(repeats.shape)

### Slide 21 — Numbers stored as text cannot be used in calculations


**Exercise.**

1. Predict which of the three sales values become NaN.
2. Run the cell.
3. Change "1,204" to "1204" and run again. What changes?


In [ ]:
raw = pd.DataFrame({"sales": ["31", "1,204", "n/a"],
                    "date": ["2024-01-05", "2024-02-11", "2024-03-02"]})

raw["sales"] = pd.to_numeric(raw["sales"], errors="coerce")
raw["date"] = pd.to_datetime(raw["date"])
print(raw)
print(raw.dtypes)

### Slide 22 — A CSV file stores characters, not types


Expected output:

```
'279839557.31'
'2022'
['80000000', '9000000']
```


In [ ]:
import csv, io

text = "contract_id,budget,year\nP-001,279839557.31,2022\n"
rows = list(csv.DictReader(io.StringIO(text)))

print(repr(rows[0]["budget"]))
print(repr(rows[0]["year"]))

print(sorted(["9000000", "80000000"]))

### Slide 23 — errors="coerce" converts failures into countable NaNs


Expected output:

```
0
start_date          973
completion_date    6414
dtype: int64
```


In [ ]:
df["budget"] = pd.to_numeric(df.budget, errors="coerce")
print(df.budget.isna().sum())        # failed to convert

for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print(df[["start_date", "completion_date"]].isna().sum())

### Slide 25 — Two date columns become one number


Expected output:

```
count    27664.0
mean       233.0
std        159.0
min        -17.0
25%        139.0
50%        202.0
75%        284.0
max       2688.0
Name: duration, dtype: float64
```


In [ ]:
df["duration"] = (df.completion_date
                   - df.start_date).dt.days

print(df.duration.describe().round())

### Slide 26 — A project that finished before it started


Expected output:

```
contract_id                   16DC0070
status                       Completed
start_date         2016-05-23 00:00:00
completion_date    2016-05-06 00:00:00
duration                         -17.0
region                     Region IV-A
Name: 27601, dtype: object
```


In [ ]:
cols = ["contract_id", "status", "start_date",
        "completion_date", "duration", "region"]
print(df.loc[df.duration < 0, cols].iloc[0])

### Slide 28 — A range check for every derived column


Expected output:

```
AssertionError: 1 negative durations
```


This cell **fails on purpose**. The `assert` is wrapped in `try`/`except` so the notebook keeps running; the message is printed instead.


In [ ]:
# fails on purpose: read the last line
df["duration"] = (df.completion_date
                   - df.start_date).dt.days

# range check, in the same cell
bad = (df.duration < 0).sum()
try:
    assert bad == 0, f"{bad} negative durations"
except AssertionError as e:
    print("AssertionError:", e)

### Slide 29 — "Cebu City" ≠ "cebu" ≠ "CEBU "


**Exercise.**

1. Predict both counts of different labels.
2. Run the cell.
3. Remove the .replace(…) line and run again. How many labels now?


In [ ]:
cities = pd.Series(["Cebu City", "cebu", "CEBU ", "Davao"])
print(cities.nunique())

clean = (cities.str.strip()
               .str.lower()
               .replace({"cebu city": "cebu"}))
print(clean.nunique())
print(clean.tolist())

### Slide 30 — Verifying labels before grouping


Expected output:

```
18
18
```


In [ ]:
print(df.region.nunique())

# normalise, then count again
print(df.region.str.strip().str.lower().nunique())

### Slide 31 — Deriving the column the question requires


**Exercise.**

1. Predict which city ranks last per 100,000 people.
2. Run the cell.
3. Compare with the ranking by raw sales. Which cities swap places?


In [ ]:
cities = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo"],
                       "region": ["Visayas", "Mindanao", "Visayas"],
                       "sales": [24, 22, 9],
                       "population": [964_000, 1_777_000, 457_000]})

cities["per_100k"] = cities["sales"] / cities["population"] * 100_000
cities["is_visayas"] = cities["region"] == "Visayas"
print(cities[["city", "sales", "per_100k", "is_visayas"]].round(2))

### Slide 33 — One-hot encoding: one yes/no column per category


Expected output:

```
city  status_Completed  status_On-Going
0    Cebu                 1                0
1   Davao                 0                1
2  Iloilo                 1                0
```


In [ ]:
t = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo"],
                  "status": ["Completed", "On-Going", "Completed"]})
print(pd.get_dummies(t, columns=["status"], dtype=int))

### Slide 35 — Selecting the columns the question requires


Expected output:

```
(3, 5) -> (3, 3)
```


In [ ]:
cities = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo"],
                       "region": ["Visayas", "Mindanao", "Visayas"],
                       "sales": [24, 22, 9],
                       "internal_id": ["A17", "B02", "C99"],
                       "notes": ["", "", ""]})

keep = cities[["city", "region", "sales"]]
print(cities.shape, "->", keep.shape)

### Slide 36 — A constant column carries no information


Expected output:

```
amount_paid     1
status          5
year           11
category       17
dtype: int64
(34079, 16) -> (34079, 15)
```


In [ ]:
print(df.nunique().sort_values().head(4))

lean = df.drop(columns=["amount_paid"])
print(df.shape, "->", lean.shape)

### Slide 37 — Concatenating adds rows; merging adds columns


Expected output:

```
5553 + 3475 -> (9028, 16)
(34079, 16) -> (34079, 17)
```


In [ ]:
# same columns, more rows
y2024 = df[df.year == 2024]
y2025 = df[df.year == 2025]
both = pd.concat([y2024, y2025])
print(len(y2024), "+", len(y2025), "->", both.shape)

# same rows, more columns: each project gets its region's count
counts = df.region.value_counts().rename("region_projects").reset_index()
wider = df.merge(counts, on="region", how="left")
print(df.shape, "->", wider.shape)

### Slide 38 — Converting a number into a countable category


Expected output:

```
duration
impossible        1
<3mo           3049
3-12mo        21101
1-2yr          3080
>2yr            433
Name: count, dtype: int64
```


In [ ]:
bins = [-9999, 0, 90, 365, 730, 9999]
labels = ["impossible", "<3mo", "3-12mo",
          "1-2yr", ">2yr"]

print(pd.cut(df.duration, bins, labels=labels, right=False)
        .value_counts().sort_index())

### Slide 40 — Comparing columns that are not in the same units


Expected output:

```
min-max: largest 1.0  median 0.026
z-score: largest 28.8  median -0.19
```


In [ ]:
x = df.budget

minmax = (x - x.min()) / (x.max() - x.min())
z = (x - x.mean()) / x.std()

print("min-max: largest", round(minmax.max(), 3), " median", round(minmax.median(), 3))
print("z-score: largest", round(z.max(), 1), " median", round(z.median(), 2))

### Slide 42 — Long and wide formats for the same numbers


Expected output:

```
(180, 3)
(18, 11) with 18 empty cells
year        2016  2017
region
Region I    5.08  5.90
Region III  8.00  9.97
```


In [ ]:
# long: one row per (region, year)
long = (df.groupby(["region", "year"])
          .budget.sum().reset_index())
print(long.shape)

# wide: regions down, years across
wide = long.pivot(index="region", columns="year",
                  values="budget")
print(wide.shape, "with", int(wide.isna().sum().sum()), "empty cells")
print((wide.loc[["Region I", "Region III"], [2016, 2017]]
       / 1e9).round(2))

### Slide 44 — Aggregation changes the grain


**Exercise.**

1. Predict the Visayas total.
2. Run the cell.
3. Change .sum() to .mean(). Which region is larger now?


In [ ]:
cities = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo"],
                       "region": ["Visayas", "Mindanao", "Visayas"],
                       "sales": [24, 22, 9]})

by_region = (cities
    .groupby("region")["sales"]
    .sum()
    .reset_index())
print(by_region)

### Slide 46 — One row per region, stated in the name


Expected output:

```
18
projects  total  typical
region
Region III                   5412  267.7     46.3
National Capital Region      3925  159.2     29.2
Region V                     2814  155.6     46.8
```


In [ ]:
by_region = df.groupby("region").agg(
    projects=("contract_id", "count"),
    total=("budget", "sum"),
    typical=("budget", "median"),
)
print(len(by_region))

top = by_region.sort_values("total", ascending=False).head(3)
top["total"] = (top.total / 1e9).round(1)        # billions of pesos
top["typical"] = (top.typical / 1e6).round(1)    # millions of pesos
print(top)

### Slide 47 — Reproducible sampling


Expected output:

```
(500, 16)
(900, 16)
```


In [ ]:
print(df.sample(500, random_state=42).shape)

# the same number from every region
per_region = df.groupby("region").sample(n=50, random_state=42)
print(per_region.shape)

### Slide 51 — Sampling large data reproducibly


**Exercise.**

1. Predict whether the two printed lists match.
2. Run the cell.
3. Delete random_state=0 from both lines and run twice. What happens?


In [ ]:
rng = np.random.default_rng(6)
budgets = pd.DataFrame({"budget": rng.integers(1, 100, 1000)})  # made-up

a = budgets.sample(n=5, random_state=0)
b = budgets.sample(n=5, random_state=0)
print(a.index.tolist())
print(b.index.tolist())

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
